# What the loop does when things go wrong

`01_one_check.ipynb` walks the happy path with a real container. This notebook
is about behaviour, so it uses a stand-in for the job — each case runs in
seconds, and the awkward situations can be staged deliberately. Everything else
is real: same database, same storage, same loop code.

Each case states its setup, its expectation, then shows what happened.

In [ ]:
import json

import pandas as pd

from recon import check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.workers import JobStatus

s3 = storage.get_s3_client()
BUCKET, _ = storage.parse_s3_path(storage.model_base_path(0))


class StandInRunner:
    """Accepts submissions and reports whatever status a case needs.

    It writes no artifacts — cases place those in storage themselves, which is
    the honest way to test a loop that believes storage over jobs.
    """

    def __init__(self):
        self.submitted, self.status = [], JobStatus.RUNNING

    def submit(self, job, payload):
        self.submitted.append((job, payload["reach_id"]))
        return f"stand-in-{payload['reach_id']}-{len(self.submitted)}"

    def poll(self, ref):
        return self.status

    def reap(self, ref):
        pass

    def logs(self, ref, tail=50):
        return "stand-in failure"


LULC = {"11": 0.04, "21": 0.04}

def reset(*reach_ids):
    """Terminal-only reaches (no dependencies), defaults seeded, storage clean."""
    with db.connect() as conn:
        conn.execute("TRUNCATE reach_network CASCADE")
        conn.execute("DELETE FROM desired_state_defaults")
        conn.execute(
            """INSERT INTO desired_state_defaults
               (sdr_commit, grid_resolution, epsg_code, dem_source, lulc_source, lulc_lookup)
               VALUES ('deadbeefcafe', 10, 5070, 's3://dem', 's3://lulc', %s)""",
            (json.dumps(LULC),))
        for rid in reach_ids:
            wkt = f"LINESTRING(0 0,{rid} 1)"  # distinct geometry -> distinct identity per reach
            conn.execute(
                "INSERT INTO reach_network (reach_id, is_terminal, terminal_reason, slope, geom) "
                "VALUES (%s, TRUE, 'outlet', 0.001, ST_GeomFromText(%s, 5070))",
                (rid, wkt))
        conn.execute("INSERT INTO desired_state (reach_id) SELECT reach_id FROM reach_network")
    for rid in reach_ids:
        clear_storage(rid)
    return StandInRunner()


def clear_storage(reach_id):
    _, base = storage.parse_s3_path(storage.model_base_path(reach_id))
    for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=base).get("Contents", []):
        s3.delete_object(Bucket=BUCKET, Key=obj["Key"])


def put_model(reach_id, manifest=True, break_identity=False):
    """Stage what a finished build leaves behind — at the PREDICTED address,
    with a manifest that passes verification (unless asked to break it)."""
    wanted = intent.effective(reach_id)
    identity_obj, ihash = identity.model_identity(wanted)
    if break_identity:
        identity_obj = {**identity_obj, "grid_resolution": 999.0}  # hash no longer matches
    model_id = f"{ihash}_N10S10E10W10"
    _, base = storage.parse_s3_path(storage.model_base_path(reach_id))
    s3.put_object(Bucket=BUCKET, Key=f"{base}/{model_id}/dem.tif", Body=b"raster")
    if manifest:
        s3.put_object(
            Bucket=BUCKET, Key=f"{base}/{model_id}/{storage.MANIFEST_FILENAME}",
            Body=json.dumps({"reach_id": reach_id, "identity_hash": ihash,
                             "identity": identity_obj,
                             "created_at": "2026-08-19T00:00:00Z"}).encode())
    return model_id


def state_of(reach_id):
    return db.one("SELECT state, model_id, model_applied_revision, desired_revision, has_gap "
                  "FROM reach_status WHERE reach_id = %s", (reach_id,))


print("ready")

## Case 1 — what already exists is adopted, not rebuilt

**Setup:** a model already sits at the address intent implies (an earlier
deployment, a restored bucket, someone's `aws s3 sync`).
**Expect:** the first check ever run adopts it and the reach is finished —
no job, no container, nothing to clean up.

In [ ]:
runner = reset(1)
put_model(1)

print("check:", check.run_check(1, runner))
print("state:", state_of(1))
print("jobs submitted:", runner.submitted)

## Case 2 — a gap gets closed, and nothing is submitted twice

**Setup:** one reach, nothing built.
**Expect:** the first check submits; a second check finds the job in flight
and does nothing; once the output appears at the predicted address, a check
adopts it; a further check is a no-op. One submission across four checks.

In [ ]:
runner = reset(2)

print("check 1:", check.run_check(2, runner))
print("check 2:", check.run_check(2, runner))

runner.status = JobStatus.SUCCEEDED
put_model(2)                                  # the job's output appears
print("status: ", jobs.status_pass(runner)[0]["action"])
print("check 3:", check.run_check(2, runner))
print("check 4:", check.run_check(2, runner))

print(f"\nsubmissions: {runner.submitted}")
print("still due:  ", [r["reach_id"] for r in queue.due_reaches()])

## Case 3 — a half-written model does not count

**Setup:** artifacts in storage, but no manifest — a build that died partway.
**Expect:** invisible. `build_model` writes `model_manifest.json` last, so the
manifest's presence is the only honest signal a build finished.

In [ ]:
runner = reset(3)
put_model(3, manifest=False)

print("observe:", observe.observe_reach(3))
print("check:  ", check.run_check(3, runner))

## Case 4 — a manifest that lies is refused

**Setup:** a manifest at the right address whose identity object does not hash
to the identity it claims — a hand-edited file, a corrupted upload, or drift
between the loop's hashing recipe and the job's.
**Expect:** not adopted. The refusal is loud and the reach is treated as
unbuilt, because adopting a hash we cannot reproduce would mean trusting a
label over the contents.

In [ ]:
runner = reset(4)
put_model(4, break_identity=True)

seen = observe.observe_reach(4)
print("adopted:", seen["found"])
print("refused:", json.dumps(seen["refused"], indent=2))
print("check:  ", check.run_check(4, runner))

## Case 5 — deleting from storage is how you undo

**Setup:** a satisfied reach; then its model is deleted from the bucket.
**Expect:** the next check finds nothing at the address, deletes the proof row
— which retracts the claim in the same statement — and rebuilds. Nothing had
to be told.

In [ ]:
runner = reset(5)
put_model(5)
check.run_check(5, runner)
print("before:", state_of(5))

clear_storage(5)
queue.request_check(5)

print("after: ", check.run_check(5, runner))
print("state: ", state_of(5))

## Case 6 — intent changes reopen the gap; reverting re-adopts for free

**Setup:** a satisfied reach. Then the deployment's `grid_resolution` changes.
**Expect:** every reach's revision bumps, the predicted address moves, and the
old model — still in the bucket — no longer counts: the proof row is deleted
and a rebuild is submitted. Reverting the change makes the *original* model
correct again, and the next check adopts it **without running any job**.

This is gap kind 4, and it needed no staleness machinery — the address moved,
that is all.

In [ ]:
runner = reset(6)
old_model = put_model(6)
check.run_check(6, runner)
print("satisfied:      ", state_of(6))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET grid_resolution = 30")
print("intent changed: ", check.run_check(6, runner), "| submissions:", len(runner.submitted))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET grid_resolution = 10")
print("reverted:       ", check.run_check(6, runner), "| submissions:", len(runner.submitted))
print("state:          ", state_of(6))

## Case 7 — failures back off, then stop

**Setup:** a runner that cannot submit at all.
**Expect:** each failure counted, the wait doubling, then the reach parked
(`halted`) for a person. Retries belong to the loop — nothing else needs to
be configured to retry, and nothing else should be.

In [ ]:
class BrokenRunner(StandInRunner):
    def submit(self, job, payload):
        raise RuntimeError("docker daemon unreachable")

reset(7)
broken = BrokenRunner()
rows = []
for attempt in range(6):
    check.run_check(7, broken)
    row = db.one("SELECT consecutive_failures, halted FROM reach_processing WHERE reach_id = 7")
    rows.append({"attempt": attempt + 1, **row,
                 "due": any(r["reach_id"] == 7 for r in queue.due_reaches())})
    with db.connect() as conn:   # skip the wait so the case finishes quickly
        conn.execute("UPDATE reach_processing SET next_retry_at = NULL WHERE reach_id = 7")

display(pd.DataFrame(rows))
processing.clear_halt(7)
print("after clear_halt, due again:", any(r["reach_id"] == 7 for r in queue.due_reaches()))

## Case 8 — a job nobody can account for

**Setup:** a job in flight whose reference means nothing any more — container
reaped, batch history aged out.
**Expect:** left alone during a grace period; then the marker is cleared with
**no failure recorded**, because we do not know that it failed. The next check
asks storage, which is the only party with an answer. The worst case is a
duplicate submission, which content-addressing makes harmless.

In [ ]:
runner = reset(8)
check.run_check(8, runner)
runner.status = JobStatus.UNKNOWN

print("still young:", jobs.status_pass(runner)[0]["action"])
with db.connect() as conn:
    conn.execute("UPDATE reach_processing SET current_step_started_at = now() - interval '30 min' WHERE reach_id = 8")
print("past grace: ", jobs.status_pass(runner)[0]["action"])
print("failures:   ", db.one("SELECT consecutive_failures FROM reach_processing WHERE reach_id = 8"))
print("next check: ", check.run_check(8, runner))

## Case 9 — a sweep settles

**Setup:** three fresh reaches (all terminal, so no dependencies here — the
cascade is `03_run_network.ipynb`'s story).
**Expect:** the first sweep submits, later sweeps record, and once satisfied a
sweep does nothing at all. A loop that has caught up is quiet.

In [ ]:
runner = reset(11, 12, 13)
rounds = []
for n in range(1, 5):
    if n == 2:
        runner.status = JobStatus.SUCCEEDED
        for rid in (11, 12, 13):
            put_model(rid)
        jobs.status_pass(runner)
    results = check.sweep(runner)
    rounds.append({"sweep": n, "checked": len(results),
                   "decisions": ", ".join(sorted({r.decision for r in results})) or "-",
                   "submitted_total": len(runner.submitted)})
display(pd.DataFrame(rounds))
display(pd.DataFrame(db.query("SELECT reach_id, state, model_id FROM reach_status ORDER BY reach_id")))

In [ ]:
with db.connect() as conn:
    conn.execute("TRUNCATE reach_network CASCADE")
    conn.execute("DELETE FROM desired_state_defaults")
for rid in (1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13):
    clear_storage(rid)
print("cleaned up")

## What these cases establish

| | |
|---|---|
| **Existing work is adopted** | intent implies the address; whatever is there and verifies, counts |
| **Work is never repeated** | the in-flight marker suppresses resubmission and cannot wedge |
| **Only storage is believed** | half-written models are invisible; lying manifests are refused |
| **Undo is deletion** | removing the model removes the proof, in one statement |
| **Intent changes are ordinary** | the address moves, the gap reopens, reverting is free |
| **Failure is bounded** | backoff, then halted, then a person |
| **Uncertainty is safe** | an unaccountable job costs at most one duplicate build |